# 🚀 AnimeKu Studio — 100% Free Video Ingestion Engine (Google Colab)
Notebook ini digunakan untuk mengunduh episode anime dari Nyaa.si, mengekstrak Subtitle Indonesia resmi, dan mengunggahnya langsung sebagai Single-File Video (MKV/MP4) utuh ke **Cloudflare R2** serta mendaftarkannya ke database **Cloudflare D1** dengan kecepatan maksimal tanpa memakan kuota internet atau komputasi pribadi Anda.

> **Tips Unlimited High-Throughput:** Mode Single-File Direct ini beroperasi dengan sangat cepat (~45-60 detik per episode) sehingga Anda dapat memproses ribuan episode setiap hari tanpa batasan kuota GPU!

In [ ]:
#@title 🔑 2. Konfigurasi Kredensial Bunny Stream dan Server VPS (1-Click Ready) { display-mode: "form" }
#@markdown > **Catatan Arsitektur:** Video hasil download diunggah langsung ke Bunny Stream (HLS Edge CDN). Server VPS IDCloudHost mengelola antrean dan database relasional.
import os

BUNNY_LIBRARY_ID = "824044" #@param {type:"string"}
BUNNY_API_KEY = "9c54f545-977e-4a9b-ac2810ab3206-1352-4634" #@param {type:"string"}
BUNNY_CDN_HOSTNAME = "vz-a43d21ad-647.b-cdn.net" #@param {type:"string"}
WORKER_BASE_URL = "http://103.52.115.44" #@param {type:"string"}
STORAGE_DRIVER = "bunny" #@param ["bunny", "r2", "hetzner"]

os.environ["BUNNY_LIBRARY_ID"] = BUNNY_LIBRARY_ID
os.environ["BUNNY_API_KEY"] = BUNNY_API_KEY
os.environ["BUNNY_CDN_HOSTNAME"] = BUNNY_CDN_HOSTNAME
os.environ["WORKER_BASE_URL"] = WORKER_BASE_URL
os.environ["STORAGE_DRIVER"] = STORAGE_DRIVER
os.environ["DEFAULT_MODE"] = "direct"
MODE = "direct"

print("✅ Kredensial Bunny Stream dan VPS Berhasil Diaktifkan!")
print(f"👉 Target Video Library : {BUNNY_LIBRARY_ID}")
print(f"👉 Target CDN Hostname  : {BUNNY_CDN_HOSTNAME}")
print(f"👉 Target Backend VPS   : {WORKER_BASE_URL}")
print("👉 Siap mendengarkan antrean job Single-File Video dari Admin Panel.")

In [ ]:
#@title 🔑 2. Konfigurasi Kredensial Cloudflare (1-Click Ready) { display-mode: "form" }
#@markdown > **Catatan Arsitektur:** Parameter judul anime, nomor episode, dan magnet **100% diatur secara dinamis dari Admin Panel**! Server Colab ini akan otomatis memproses anime apapun yang Anda antrekan dari Admin Panel.
import os

try:
    from google.colab import userdata
    R2_ACCESS_KEY = userdata.get("R2_ACCESS_KEY")
    R2_SECRET_KEY = userdata.get("R2_SECRET_KEY")
except Exception:
    R2_ACCESS_KEY = None
    R2_SECRET_KEY = None

CF_ACCOUNT_ID = "5b5d0b730a79c86cf659a0d01f86ba9f" #@param {type:"string"}
CF_DATABASE_ID = "b51144f3-8050-4349-815c-ecb3402510f7" #@param {type:"string"}
R2_ACCESS_KEY = R2_ACCESS_KEY or "d049e38782fa0463dda2dc0129cb6fa5" #@param {type:"string"}
R2_SECRET_KEY = R2_SECRET_KEY or "100d2ff77386ce351f80b82ec300b6ee8b1a71fd271ac73c6f12c829d67a3f87" #@param {type:"string"}
R2_BUCKET_NAME = "animekuapp-media" #@param {type:"string"}
STORAGE_DRIVER = "r2" #@param ["r2", "hetzner"]

os.environ["CF_ACCOUNT_ID"] = CF_ACCOUNT_ID
os.environ["CF_DATABASE_ID"] = CF_DATABASE_ID
os.environ["R2_ACCESS_KEY"] = R2_ACCESS_KEY
os.environ["R2_SECRET_KEY"] = R2_SECRET_KEY
os.environ["R2_BUCKET_NAME"] = R2_BUCKET_NAME
os.environ["STORAGE_DRIVER"] = STORAGE_DRIVER
os.environ["DEFAULT_MODE"] = "direct"
MODE = "direct"

print("✅ Kredensial Cloudflare R2 & D1 Berhasil Diaktifkan!")
print("👉 Siap mendengarkan antrean job Single-File Video dari Admin Panel.")

In [ ]:
#@title 🚀 3. Jalankan Always-On Worker Daemon (Otomatis Mendengarkan Admin Panel) { display-mode: "form" }
#@markdown Klik tombol ▶️ di bawah. Server Colab akan standby 24/7 dan otomatis memproses anime apapun yang Anda dispatch dari Admin Panel.
import os, subprocess, sys
from IPython.display import Javascript, display

# Fallback kredensial otomatis jika Cell 2 belum dijalankan
if not os.environ.get("R2_ACCESS_KEY"):
    os.environ["CF_ACCOUNT_ID"] = "5b5d0b730a79c86cf659a0d01f86ba9f"
    os.environ["CF_DATABASE_ID"] = "b51144f3-8050-4349-815c-ecb3402510f7"
    os.environ["R2_ACCESS_KEY"] = "d049e38782fa0463dda2dc0129cb6fa5"
    os.environ["R2_SECRET_KEY"] = "100d2ff77386ce351f80b82ec300b6ee8b1a71fd271ac73c6f12c829d67a3f87"
    os.environ["R2_BUCKET_NAME"] = "animekuapp-media"
    os.environ["STORAGE_DRIVER"] = "r2"

os.environ["DEFAULT_MODE"] = "direct"

# 1. Anti-Disconnect Heartbeat: Mensimulasikan heartbeat DOM agar Colab tidak tidur/idle
display(Javascript("""
  function ColabHeartbeat() {
    console.log("[AnimeKu Daemon] Heartbeat Keep-Alive: " + new Date().toLocaleTimeString());
    var btn = document.querySelector("colab-connect-button");
    if (btn) btn.click();
  }
  if (!window.animekuHeartbeatInterval) {
    window.animekuHeartbeatInterval = setInterval(ColabHeartbeat, 60000);
    console.log("🛡️ Anti-Sleep Heartbeat Aktif (Interval 60s)!");
  }
"""))

print("🟢 [ALWAYS-ON DAEMON] Server Colab Aktif Standby Mendengarkan Admin Panel!")
print("   👉 Anda bisa minimize tab ini atau buka HP.")
print("   👉 Kapanpun Anda membuat job di Admin Panel (anime apapun: Solo Leveling, One Piece, dsb.),")
print("   👉 Server ini akan otomatis langsung mengambil data job tersebut & mengunggah Single-File Video ke R2!\n")

# Pastikan selalu berada di direktori runner animeku-worker
if os.path.exists("/content/animeku-worker"):
    os.chdir("/content/animeku-worker")

print("🔄 Memperbarui script runner ke commit terbaru dari GitHub...")
subprocess.run("git fetch origin main --quiet && git reset --hard origin/main --quiet", shell=True)

cmd = "node scripts/ingest.js worker-daemon --mode direct --worker google-colab"
print(f"⚡ Menjalankan Daemon: {cmd}\n")

# 2. Live stream output seketika ke terminal Colab (Zero-Buffering)
proc = subprocess.Popen(cmd, shell=True)
proc.wait()

if proc.returncode == 0:
    print("\n👋 Daemon dihentikan secara normal.")
else:
    print(f"\n❌ Proses terhenti dengan error code {proc.returncode}.")